# H3b: Partial Equalization in a Toy Deep-Linear Setting

This notebook is the presentation/analysis companion to
`experiments/Tier_1_Core_Mechanism_Tests/H3b_PARTIAL_EQUALIZATION/run_experiment.py`.
It **imports and executes the script's `run_experiment()` function** so that the script remains the
single source of truth for the actual experiment logic.

## Truthful scope of this first-pass study

- This is a **toy deep-linear final-loss experiment**.
- It probes whether **partial singular-value equalization** can recover **Muon-like final-loss behavior**.
- It does **not** directly measure causal "singular-value rescue" during training; there is no per-step
  spectral logging or direct mechanism attribution in this pass.
- The notebook therefore reports **what the current code actually measures**: learning-rate sweeps,
  per-seed final losses, summary statistics, gap-captured heuristics, and a real `k = d` equivalence check.

## Partial-polar construction used by the script

For a gradient matrix $G = U\,\mathrm{diag}(\sigma)\,V^T$ and a chosen $k$:

- the top-$k$ singular values are set to $1$;
- the remaining tail singular values keep their relative ratios but are rescaled as
  $\sigma_i / \|\sigma_{\mathrm{tail}}\| \cdot \sqrt{d-k}$.

This means the notebook studies a controlled interpolation between:

- **small $k$**: only a few leading singular directions are equalized,
- **$k=d$**: the full polar factor $UV^T$.

> Important distinction: `k=0` in `partial_polar()` would be a **per-layer, pre-momentum** Frobenius-normalized
> gradient transform. That is **not** the same optimizer as the script's **NormSGD baseline**, which accumulates
> raw momentum first and normalizes the momentum before each step.


In [ ]:
import os
import sys
import time
import pathlib
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Markdown
except ModuleNotFoundError:
    def display(obj):
        print(obj)

    def Markdown(text):
        return text

pd.options.display.float_format = '{:.6e}'.format
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25


## 1. Notebook-safe path resolution and script import

This notebook avoids `__file__` and instead searches upward from the current working directory until it finds
this experiment's script. That makes the notebook clean-kernel safe under both JupyterLab and batch execution
(for example via `nbconvert --execute`).


In [ ]:
def find_repo_root(start=None):
    start = pathlib.Path(start or pathlib.Path.cwd()).resolve()
    relative_script = pathlib.Path(
        'experiments/Tier_1_Core_Mechanism_Tests/H3b_PARTIAL_EQUALIZATION/run_experiment.py'
    )
    for candidate in [start, *start.parents]:
        if (candidate / relative_script).exists():
            return candidate
    raise FileNotFoundError(
        'Could not locate repo root containing ' + str(relative_script)
    )


REPO_ROOT = find_repo_root()
EXPERIMENT_DIR = REPO_ROOT / 'experiments' / 'Tier_1_Core_Mechanism_Tests' / 'H3b_PARTIAL_EQUALIZATION'
SCRIPT_PATH = EXPERIMENT_DIR / 'run_experiment.py'
NOTEBOOK_PATH = EXPERIMENT_DIR / 'run_experiment.ipynb'

spec = importlib.util.spec_from_file_location('h3b_partial_equalization', SCRIPT_PATH)
h3b = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = h3b
spec.loader.exec_module(h3b)

display(Markdown(
    '\n'.join([
        '**Counterpart identity**',
        f'- repo root: `{REPO_ROOT}`',
        f'- script source of truth: `{SCRIPT_PATH}`',
        f'- notebook: `{NOTEBOOK_PATH}`',
    ])
))


## 2. Reproducibility, runtime, and execution mode

By default, the notebook uses the script's full toy configuration. For automated smoke tests, setting
`H3B_SMOKE=1` switches to a much smaller configuration while preserving the same code path and the same
analysis structure.


In [ ]:
SMOKE_MODE = os.environ.get('H3B_SMOKE', '0') == '1'

CONFIG_OVERRIDES = None
if SMOKE_MODE:
    CONFIG_OVERRIDES = {
        'num_steps': 40,
        'num_seeds': 3,
        'lr_tuning_seeds': 2,
        'lr_candidates': [0.02, 0.01, 0.005],
        't3_matrix_samples': 2,
    }

planned_config = h3b.resolve_config(CONFIG_OVERRIDES)
planned_seeds = h3b.make_seed_schedule(CONFIG_OVERRIDES)
planned_tuning_seeds = planned_seeds[:planned_config['lr_tuning_seeds']]

mode_label = 'SMOKE MODE (reduced config for execution checks)' if SMOKE_MODE else 'FULL DEFAULT TOY EXPERIMENT'
print(mode_label)
print('=' * len(mode_label))
print(f'script path:           {SCRIPT_PATH}')
print(f'notebook path:         {NOTEBOOK_PATH}')
print(f'dimension:             {planned_config["dim"]}')
print(f'layers:                {planned_config["num_layers"]}')
print(f'steps:                 {planned_config["num_steps"]}')
print(f'batch size:            {planned_config["batch_size"]}')
print(f'evaluation seeds:      {planned_seeds}')
print(f'LR-tuning seeds:       {planned_tuning_seeds}')
print(f'k values:              {planned_config["k_values"]}')
print(f'LR candidates:         {planned_config["lr_candidates"]}')
print()
print('Interpretation notes:')
print('- lower final loss is better')
print('- error bars below are SEM across seeds unless stated otherwise')
print('- gap captured is measured relative to the mean NormSGD-to-k=d gap')
print('- negative gap captured means a method was worse than NormSGD in that run')


## 3. Run the script-defined experiment

The script returns a structured results dictionary containing configuration, seed schedules, learning-rate sweep
records, per-seed final losses, summaries, derived metrics, and explicit T1/T2/T3 test outputs.


In [ ]:
wall_start = time.time()
results = h3b.run_experiment(config_overrides=CONFIG_OVERRIDES, verbose=False)
wall_runtime = time.time() - wall_start

cfg = results['config']
seeds = results['seeds']
phase1 = results['phase1']
phase2 = results['phase2']
derived = results['derived_metrics']
tests = results['tests']

print(f'run_experiment() reported runtime: {results["runtime_sec"]:.2f} s')
print(f'observed notebook wall time:       {wall_runtime:.2f} s')
print(f'full seed list:                    {seeds["all"]}')
print(f'LR-tuning seed list:               {seeds["lr_tuning"]}')
print(f'Muon-like mean-loss threshold:     {derived["muon_like_loss_threshold"]:.6e}')


## 4. Summary tables

These tables expose the main quantities the original script printed informally, but now in a reusable and auditable
format suitable for side-by-side reading with the script.


In [ ]:
partial_rows = [row for row in derived['summary_rows'] if row['method'] == 'partial_equalization']
partial_df = pd.DataFrame(partial_rows).sort_values('k').reset_index(drop=True)
baseline_df = pd.DataFrame([
    row for row in derived['summary_rows']
    if row['method'] in {'norm_sgd', 'muon_reference'}
]).reset_index(drop=True)

performance_df = partial_df[[
    'k', 'best_lr', 'mean_loss', 'std_loss', 'sem_loss', 'n_finite',
    'gap_captured_pct', 'vs_normsgd', 'vs_muon'
]].copy()
performance_df.columns = [
    'k', 'best_lr', 'mean_loss', 'std_loss', 'sem_loss', 'n_finite',
    'gap_captured_pct', 'normsgd_over_loss', 'loss_over_kdim'
]

best_lr_rows = []
for k in cfg['k_values']:
    rec = phase1['partial'][k]
    best_lr_rows.append({
        'method': f'partial_k={k}',
        'k': k,
        'best_lr': rec['best_lr'],
        'tuning_mean_loss': rec['best_mean_loss'],
    })
best_lr_rows.append({
    'method': 'norm_sgd',
    'k': np.nan,
    'best_lr': phase1['norm_sgd']['best_lr'],
    'tuning_mean_loss': phase1['norm_sgd']['best_mean_loss'],
})
best_lr_df = pd.DataFrame(best_lr_rows)

baseline_view = baseline_df[[
    'method', 'best_lr', 'mean_loss', 'std_loss', 'sem_loss', 'n_finite', 'gap_captured_pct', 'vs_muon'
]].copy()
baseline_view.columns = [
    'method', 'best_lr', 'mean_loss', 'std_loss', 'sem_loss', 'n_finite', 'gap_captured_pct', 'loss_over_kdim'
]

print('Mean final-loss summary for partial equalization sweep')
display(performance_df)
print('Baselines and explicit Muon reference')
display(baseline_view)
print('Selected best learning rates from phase-1 tuning')
display(best_lr_df)


**Reading the tables.**

- `mean_loss`, `std_loss`, and `sem_loss` come from the full evaluation seeds.
- `gap_captured_pct` is defined relative to the **mean** gap between NormSGD and the `k=d` partial-equalization endpoint.
- Values **below 0%** mean worse-than-NormSGD performance.
- Values **near 100%** mean close to the `k=d` partial-equalization endpoint under this metric.
- The explicit `muon_reference` row is included only for the `T3` sanity check; it verifies that the `k=d` code path
  actually matches an explicit polar-factor implementation.


In [ ]:
fig, ax = plt.subplots()

x = partial_df['k'].to_numpy(dtype=float)
y = partial_df['mean_loss'].to_numpy(dtype=float)
yerr = partial_df['sem_loss'].fillna(0.0).to_numpy(dtype=float)

ax.errorbar(x, y, yerr=yerr, marker='o', linewidth=2, capsize=4, label='partial equalization (mean ± SEM)')
ax.axhline(phase2['norm_sgd']['summary']['mean'], color='tab:red', linestyle='--', label='NormSGD mean')
ax.axhline(phase2['muon_reference']['summary']['mean'], color='tab:green', linestyle=':', label='explicit Muon mean')
ax.scatter([cfg['dim']], [phase2['partial'][cfg['dim']]['summary']['mean']], color='black', zorder=5, label='k=d partial endpoint')

ax.set_xscale('log', base=2)
ax.set_xticks(cfg['k_values'])
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
if np.all(np.isfinite(y)) and np.all(y > 0):
    ax.set_yscale('log')
ax.set_xlabel('number of equalized singular values (k)')
ax.set_ylabel('mean final loss')
ax.set_title('Final mean loss vs k (full evaluation; mean ± SEM)')
ax.legend(loc='best')
fig.tight_layout()
plt.show()


**Interpretation.**

This figure should be read strictly as an **optimization-outcome plot**. Lower loss means better final performance in
this toy task, but the plot alone does not identify *why* a given $k$ performs better. Any Muon/RG interpretation must
therefore remain downstream and explicitly labeled as conjectural.


In [ ]:
fig, ax = plt.subplots()

gap_values = partial_df['gap_captured_pct'].to_numpy(dtype=float)
ax.plot(x, gap_values, marker='o', linewidth=2)
ax.axhline(0.0, color='black', linewidth=1)
ax.axhline(50.0, color='tab:orange', linestyle='--', label='T1 threshold: 50%')
ax.axhline(80.0, color='tab:purple', linestyle='--', label='T2 threshold: 80%')
ax.axhline(100.0, color='tab:green', linestyle=':', label='k=d endpoint')

ax.set_xscale('log', base=2)
ax.set_xticks(cfg['k_values'])
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.set_xlabel('number of equalized singular values (k)')
ax.set_ylabel('% of NormSGD→k=d mean-loss gap captured')
ax.set_title('Gap captured vs k')
ax.legend(loc='best')
fig.tight_layout()
plt.show()


**Interpretation.**

The gap-captured view is useful because it normalizes performance relative to the observed NormSGD baseline and the
observed `k=d` endpoint. However, it remains a heuristic summary of **mean final-loss differences**. It is not a direct
measurement of singular-value dynamics, and negative values should be read literally rather than clipped away.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for k in cfg['k_values']:
    records = phase1['partial'][k]['lr_records']
    lr = np.array([r['lr'] for r in records], dtype=float)
    mean_loss = np.array([r['summary']['mean'] for r in records], dtype=float)
    mean_loss[~np.isfinite(mean_loss)] = np.nan
    ax.plot(lr, mean_loss, marker='o', linewidth=1.5, label=f'k={k}')

norm_lr = np.array([r['lr'] for r in phase1['norm_sgd']['lr_records']], dtype=float)
norm_mean = np.array([r['summary']['mean'] for r in phase1['norm_sgd']['lr_records']], dtype=float)
norm_mean[~np.isfinite(norm_mean)] = np.nan
ax.plot(norm_lr, norm_mean, marker='s', linewidth=2, linestyle='--', color='black', label='NormSGD')

ax.set_xscale('log')
finite_lr_sweep_values = np.concatenate([
    norm_mean[np.isfinite(norm_mean)],
    np.array([value for value in partial_df['mean_loss'].to_numpy(dtype=float) if np.isfinite(value)], dtype=float),
])
if finite_lr_sweep_values.size and np.all(finite_lr_sweep_values > 0):
    ax.set_yscale('log')
ax.set_xlabel('learning rate')
ax.set_ylabel('mean final loss on LR-tuning seeds')
ax.set_title('Phase-1 learning-rate sweep curves')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5))
fig.tight_layout()
plt.show()


## 5. Explicit T1/T2/T3 reporting

The original script's `T3` check was tautological. In this pass, `T3` is replaced by a real sanity check with two parts:

1. **matrix-level equivalence**: verify `partial_polar(G, d)` matches an explicit `polar_factor(G)` on random matrices;
2. **training-level equivalence**: verify that the `k=d` training path matches an explicit Muon-reference training function.


In [ ]:
test_rows = [
    {
        'test': 'T1',
        'passed': tests['T1']['passed'],
        'question': tests['T1']['question'],
        'key_result': f"gap captured by k=1 = {tests['T1']['gap_captured_pct']:.2f}%",
        'interpretable': tests['T1']['gap_interpretable'],
    },
    {
        'test': 'T2',
        'passed': tests['T2']['passed'],
        'question': tests['T2']['question'],
        'key_result': (
            f"first k over 80% = {tests['T2']['knee_k']} (gap = {tests['T2']['knee_gap_captured_pct']:.2f}%)"
            if tests['T2']['knee_k'] is not None else 'no tested k exceeded 80% gap captured'
        ),
        'interpretable': tests['T2']['gap_interpretable'],
    },
    {
        'test': 'T3',
        'passed': tests['T3']['passed'],
        'question': tests['T3']['question'],
        'key_result': 'matrix and training equivalence both checked explicitly',
        'interpretable': True,
    },
]

test_df = pd.DataFrame(test_rows)
display(test_df)

matrix_equiv_df = pd.DataFrame([{
    'samples': tests['T3']['matrix_equivalence']['samples'],
    'max_abs_error_fro': tests['T3']['matrix_equivalence']['max_abs_error_fro'],
    'max_rel_error_fro': tests['T3']['matrix_equivalence']['max_rel_error_fro'],
    'rtol': tests['T3']['matrix_equivalence']['rtol'],
    'passed': tests['T3']['matrix_equivalence']['passed'],
}])
training_equiv_df = pd.DataFrame([{
    'max_abs_diff': tests['T3']['training_equivalence']['max_abs_diff'],
    'max_rel_diff': tests['T3']['training_equivalence']['max_rel_diff'],
    'rtol': tests['T3']['training_equivalence']['rtol'],
    'atol': tests['T3']['training_equivalence']['atol'],
    'inf_pattern_match': tests['T3']['training_equivalence']['inf_pattern_match'],
    'allclose': tests['T3']['training_equivalence']['allclose'],
}])

print('T3 matrix-level equivalence diagnostics')
display(matrix_equiv_df)
print('T3 training-level equivalence diagnostics')
display(training_equiv_df)

muon_like_df = pd.DataFrame([{
    'muon_like_rtol': derived['muon_like_loss_rtol'],
    'muon_like_threshold': derived['muon_like_loss_threshold'],
    'smallest_k_within_muon_like_mean_loss': derived['smallest_k_within_muon_like_mean_loss'],
}])
print('Smallest tested k whose mean loss falls within the Muon-like tolerance band')
display(muon_like_df)


## 6. Calibrated conclusion

The conclusion below is generated from the actual run outputs. It is intentionally conservative: it states what this
notebook *observed* in the current final-loss study and avoids claiming a direct mechanistic proof that is not measured.


In [ ]:
smallest_k = derived['smallest_k_within_muon_like_mean_loss']
muon_tol_pct = 100 * derived['muon_like_loss_rtol']
muon_loss = derived['muon_loss_mean']
norm_loss = derived['normsgd_loss_mean']

if smallest_k is None:
    recovery_statement = (
        f'No tested k fell within {muon_tol_pct:.1f}% of the k=d mean-loss endpoint. '
        'In this run, partial equalization did not recover Muon-like behavior under that threshold.'
    )
elif smallest_k < cfg['dim']:
    recovery_statement = (
        f'Yes—partial equalization reached Muon-like mean final loss before full equalization: '
        f'the smallest tested k within {muon_tol_pct:.1f}% of the k=d endpoint was k={smallest_k}.'
    )
else:
    recovery_statement = (
        f'Only the full endpoint k=d satisfied the {muon_tol_pct:.1f}% Muon-like criterion in this run. '
        'So partial equalization did not clearly recover Muon-like behavior before full equalization.'
    )

if np.isfinite(norm_loss) and np.isfinite(muon_loss):
    headline_gap = f'NormSGD mean loss = {norm_loss:.6e}; k=d partial endpoint mean loss = {muon_loss:.6e}.'
else:
    headline_gap = 'The baseline/end-point gap was not finite, so gap-based interpretation is limited.'

t1_status = 'PASS' if tests['T1']['passed'] else 'FAIL'
t2_status = 'PASS' if tests['T2']['passed'] else 'FAIL'
t3_status = 'PASS' if tests['T3']['passed'] else 'FAIL'
t3_match_word = 'matched' if tests['T3']['passed'] else 'did not fully match'

if tests['T2']['knee_k'] is not None:
    t2_detail = (
        f'the first tested knee was at k={tests["T2"]["knee_k"]} '
        f'with gap captured {tests["T2"]["knee_gap_captured_pct"]:.2f}%.'
    )
else:
    t2_detail = 'no tested k exceeded the 80% gap threshold.'

conclusion_lines = [
    '### Run-linked conclusion',
    '',
    f'- **Main empirical outcome.** {recovery_statement}',
    f'- **Observed baseline/end-point values.** {headline_gap}',
    f'- **T1 result.** {t1_status}: k=1 captured {tests["T1"]["gap_captured_pct"]:.2f}% of the observed mean-loss gap.',
    f'- **T2 result.** {t2_status}: {t2_detail}',
    f'- **T3 result.** {t3_status}: the `k=d` code path {t3_match_word} the explicit polar-factor / Muon reference under the implemented numerical checks.',
    '- **Scope caveat.** This remains a **final-loss-only toy study**. It can support claims about whether top-k equalization recovered the *observed optimization outcome* in this setup, but it does **not** by itself prove a direct singular-value-rescue mechanism.',
]

display(Markdown('\n'.join(conclusion_lines)))
